# MPD validation: CO₂/propane in MOR (Sharma et al. 2023)

Reference distribution from TMMC (`tmmc/`, stitched by `mpd/build_distribution.py`); RUPTURA runs in `runs/`.
Run `mpd/build_distribution.py 300` (and `350`), `runs/generate.py`, and `runs/run.py` first.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({"font.size": 14, "axes.labelsize": 14, "axes.titlesize": 16, "legend.fontsize": 14,
                     "xtick.labelsize": 12, "ytick.labelsize": 12, "axes.labelpad": 6.0, "axes.grid": False})
HERE = Path.cwd()
RUNS = HERE / "runs"
FIGURES = HERE / "figures"
FIGURES.mkdir(exist_ok=True)
COMPONENTS = {"CO2": 1, "C3H8": 2}
LABELS = {"CO2": "CO$_2$", "C3H8": "C$_3$H$_8$"}
TABLE_3 = {"CO2": [(1.60, 2.617e-4), (3.62, 4.859e-7)], "C3H8": [(1.73, 6.506e-9), (1.09, 2.376e-4)]}

In [ ]:
def mixture(run, component):
    """Pressure [Pa], pure and mixture loading [mol/kg] of one component of a RUPTURA mixture prediction."""
    data = np.loadtxt(RUNS / run / f"component_{COMPONENTS[component]}_{component}.data", ndmin=2)
    return data[:, 0], data[:, 1], data[:, 2]


def outlet(run, component):
    """Time [s] and outlet concentration relative to the feed of one component of a RUPTURA breakthrough."""
    data = np.loadtxt(RUNS / run / f"component_{COMPONENTS[component]}_{component}.data")
    outlet_rows = data[data[:, 2] == data[:, 2].max()]
    feed = 0.1 * 1.0e5 / (8.314462618 * 300.0)
    return outlet_rows[:, 1] * 60.0, outlet_rows[:, 3] / feed


def distribution(temperature):
    """ln Pi[n_CO2, n_C3H8] at the TMMC fugacity, the reweighting energy U - U_intra [K], and run settings."""
    data = np.load(HERE / "mpd" / f"lnpi_T{temperature:.0f}K.npz")
    return data["lnpi"], data["energy"] - data["intramolecular"], float(data["reference_pressure"]), \
        float(data["framework_mass"])


def reweight(temperature, reference_temperature, pressures, mole_fraction_co2, with_temperature_term):
    """Loadings [mol/kg] of CO2 and propane from the reference_temperature distribution reweighted to temperature.

    First-order reweighting at fixed fugacity; with_temperature_term adds N ln(T_ref/T) as RUPTURA does (B-PHY-1).
    """
    lnpi, energy, reference_pressure, framework_mass = distribution(reference_temperature)
    n = np.arange(lnpi.shape[0])[:, None]
    m = np.arange(lnpi.shape[1])[None, :]
    populated = np.isfinite(lnpi)
    log_weight = lnpi - (1.0 / temperature - 1.0 / reference_temperature) * np.where(populated, energy, 0.0)
    if with_temperature_term:
        log_weight = log_weight + (n + m) * np.log(reference_temperature / temperature)
    loadings = []
    for pressure in pressures:
        weight = log_weight + n * np.log(mole_fraction_co2 * pressure / reference_pressure) \
            + m * np.log((1.0 - mole_fraction_co2) * pressure / reference_pressure)
        weight = np.where(populated, weight, -np.inf)
        probability = np.exp(weight - weight.max())
        probability /= probability.sum()
        loadings.append([(probability * n).sum(), (probability * m).sum()])
    return np.array(loadings) / (6.02214076e23 * framework_mass)

## Pure components: MPD against the Table 3 fits

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for color, component in zip(["C0", "C1"], COMPONENTS):
    pressure, _, loading = mixture(f"mixture_pure_{component}_MPD", component)
    langmuir = sum(q * k * pressure / (1.0 + k * pressure) for q, k in TABLE_3[component])
    ax.plot(pressure, loading, "-", color=color, label=f"{LABELS[component]}, MPD")
    ax.plot(pressure[::4], langmuir[::4], "--o", color=color, markerfacecolor="none",
            label=f"{LABELS[component]}, Table 3 fit")
ax.set_xscale("log")
ax.set_xlabel("Fugacity, $f$ / Pa")
ax.set_ylabel("Loading, $q$ / mol kg$^{-1}$")
ax.legend()
fig.savefig(FIGURES / "pure_components_mpd_vs_table3.pdf", dpi=300, bbox_inches="tight")

## Equimolar mixture: MPD, SIAST, and IAST

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for color, component in zip(["C0", "C1"], COMPONENTS):
    pressure, _, loading = mixture("mixture_equimolar_MPD", component)
    ax.plot(pressure, loading, "-", color=color, label=f"{LABELS[component]}, MPD")
    pressure, _, loading = mixture("mixture_equimolar_SIAST", component)
    ax.plot(pressure[::4], loading[::4], "--o", color=color, markerfacecolor="none", label=f"{LABELS[component]}, SIAST")
    pressure, _, loading = mixture("mixture_equimolar_IAST", component)
    ax.plot(pressure[::4], loading[::4], ":^", color=color, markerfacecolor="none", label=f"{LABELS[component]}, IAST")
ax.set_xscale("log")
ax.set_xlabel("Total pressure, $P$ / Pa")
ax.set_ylabel("Loading, $q$ / mol kg$^{-1}$")
ax.legend(ncol=2)
fig.savefig(FIGURES / "equimolar_mixture_mpd_siast_iast.pdf", dpi=300, bbox_inches="tight")

## Breakthrough (Fig. 10a of the paper)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for color, component in zip(["C0", "C1"], COMPONENTS):
    for method, style in [("MPD", "-"), ("SIAST", "--"), ("IAST", ":")]:
        time, relative = outlet(f"breakthrough_{method}", component)
        ax.plot(time, relative, style, color=color, label=f"{LABELS[component]}, {method}")
ax.set_xlabel("Time, $t$ / s")
ax.set_ylabel("Outlet concentration, $c/c_0$ / [-]")
ax.legend(ncol=2)
fig.savefig(FIGURES / "breakthrough_mpd_siast_iast.pdf", dpi=300, bbox_inches="tight")

In [ ]:
for method in ["MPD", "SIAST", "IAST"]:
    print(f"{method:6s} {float((RUNS / f'breakthrough_{method}' / 'wall_time.txt').read_text()):10.1f} s wall time")

## Temperature reweighting (B-PHY-1)

The 300 K distribution reweighted to 350 K at first order, with and without the N ln(T_ref/T) term, against the
350 K TMMC (equimolar).

In [ ]:
pressures = np.logspace(2, 8, 25)
fig, ax = plt.subplots(figsize=(8, 6))
direct = reweight(350.0, 350.0, pressures, 0.5, True)
with_term = reweight(350.0, 300.0, pressures, 0.5, True)
without_term = reweight(350.0, 300.0, pressures, 0.5, False)
for index, (color, component) in enumerate(zip(["C0", "C1"], COMPONENTS)):
    ax.plot(pressures, direct[:, index], "--o", color=color, markerfacecolor="none",
            label=f"{LABELS[component]}, TMMC 350 K")
    ax.plot(pressures, with_term[:, index], "-", color=color, label=f"{LABELS[component]}, 300 K, with $N\\ln(T_\\mathrm{{ref}}/T)$")
    ax.plot(pressures, without_term[:, index], ":", color=color, label=f"{LABELS[component]}, 300 K, without")
ax.set_xscale("log")
ax.set_xlabel("Total pressure, $P$ / Pa")
ax.set_ylabel("Loading, $q$ / mol kg$^{-1}$")
ax.legend(fontsize=12)
fig.savefig(FIGURES / "temperature_reweighting_350K.pdf", dpi=300, bbox_inches="tight")

In [ ]:
# RUPTURA's own reweighting (with the term) against the Python one, and against the 350 K distribution
for run in ["mixture_equimolar_MPD_T350K_ref300K", "mixture_equimolar_MPD_T350K_ref350K"]:
    pressure, _, co2 = mixture(run, "CO2")
    python = reweight(350.0, float(run[-4:-1]), pressure, 0.5, True)[:, 0]
    print(f"{run}: max |RUPTURA - Python| = {np.abs(co2 - python).max():.2e} mol/kg")

## Reference distribution

In [ ]:
lnpi, _, reference_pressure, _ = distribution(300.0)
n = np.arange(lnpi.shape[0])[:, None]
m = np.arange(lnpi.shape[1])[None, :]
shifted = lnpi + (n + m) * np.log(1.0e6 / reference_pressure)
shifted -= shifted[np.isfinite(shifted)].max()
fig, ax = plt.subplots(figsize=(8, 6))
image = ax.imshow(np.where(shifted > -60.0, shifted, np.nan).T, origin="lower", aspect="auto", cmap="viridis")
fig.colorbar(image, ax=ax, label=r"$\ln \Pi$ at $f = 10^{6}$ Pa / [-]")
ax.set_xlabel("CO$_2$ molecules, $n_\\mathrm{CO_2}$ / [-]")
ax.set_ylabel("Propane molecules, $n_\\mathrm{C_3H_8}$ / [-]")
fig.savefig(FIGURES / "lnpi_300K.pdf", dpi=300, bbox_inches="tight")